In [1]:
import pandas as pd
import numpy as np
# import boto3
# import s3fs
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)  # so V1–V28 don't get truncated

In [2]:
df = pd.read_csv("s3://fraud-detection-620794556779-ap-southeast-2-an/datasets/creditcard.csv")

In [6]:
print(df.columns)

In [3]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nMissing values:", df.isnull().sum().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [4]:
counts = df["Class"].value_counts()
pct = df["Class"].value_counts(normalize=True) * 100

print(counts)
print(f"\nFraud rate: {pct[1]:.3f}%")
print(f"A model predicting 'never fraud' would be {pct[0]:.3f}% accurate")

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x="Class", data=df, ax=ax[0])
ax[0].set_title("Class counts (linear scale)")
ax[0].set_xticklabels(["Legit (0)", "Fraud (1)"])

sns.countplot(x="Class", data=df, ax=ax[1])
ax[1].set_yscale("log")
ax[1].set_title("Class counts (log scale)")
ax[1].set_xticklabels(["Legit (0)", "Fraud (1)"])

plt.tight_layout()
plt.savefig("eda_class_imbalance.png", dpi=120, bbox_inches="tight")
plt.show()

In [7]:
print(df.groupby("Class")["Amount"].describe())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df[df.Class == 0], x="Amount", bins=50, ax=ax[0])
ax[0].set_title("Legit transaction amounts")
ax[0].set_xlim(0, 500)

sns.histplot(data=df[df.Class == 1], x="Amount", bins=50, ax=ax[1], color="crimson")
ax[1].set_title("Fraud transaction amounts")
ax[1].set_xlim(0, 500)

plt.tight_layout()
plt.savefig("eda_amount_by_class.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
df["hour"] = (df["Time"] / 3600) % 24   # rough hour-of-day proxy

fraud_by_hour = df.groupby(df["hour"].astype(int))["Class"].mean() * 100

plt.figure(figsize=(10, 4))
fraud_by_hour.plot(kind="bar")
plt.ylabel("Fraud rate (%)")
plt.xlabel("Hour (approx, from elapsed time)")
plt.title("Fraud rate by approximate hour")
plt.tight_layout()
plt.savefig("eda_fraud_by_hour.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
v_cols = [f"V{i}" for i in range(1, 29)]

# Difference in mean value between fraud and legit, per component
separation = (
    df[df.Class == 1][v_cols].mean() - df[df.Class == 0][v_cols].mean()
).abs().sort_values(ascending=False)

print("Top separating components:\n", separation.head(6))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for cls, a, title in [(0, ax[0], "Legit"), (1, ax[1], "Fraud")]:
    sns.histplot(df[df.Class == cls]["V14"], bins=50, ax=a,
                 color="steelblue" if cls == 0 else "crimson")
    a.set_title(f"V14 distribution — {title}")

plt.tight_layout()
plt.savefig("eda_v14_separation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
df = df.drop(columns=["hour"])